In [1]:
!nvidia-smi

Tue Sep 29 09:35:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


STEP 2.1 — Install our libraries

In [4]:
!pip install -q -U transformers accelerate datasets sentence-transformers faiss-cpu peft trl

In [5]:
import transformers
import datasets
import peft
import trl
import sentence_transformers
import torch

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("PyTorch:", torch.__version__)

Transformers: 5.17.0
Datasets: 5.0.1
PEFT: 0.21.1
TRL: 1.14.0
Sentence Transformers: 6.1.0
PyTorch: 2.11.0+cu128


STEP 2.2 — Load our first LLM

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Model loaded successfully!")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded successfully!


STEP 2.3 — Ask our first question

In [7]:
messages = [
    {
        "role": "user",
        "content": "What are the common symptoms of tomato early blight?"
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.7,
    top_p=0.8,
    do_sample=True
)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

Tomato early blight (also known as **tomato bacterial wilt** or **tomato wilt disease**) is a fungal infection caused by *delayed-type necrosis of the tomato plant*, primarily affecting the **lower parts of the plant**. The common symptoms include:

1. **Mottling or yellowing** of the leaves at the base of the plant.
2. **Browning or darkening** of the leaf edges.
3. **Dark, water-soaked spots** or **pores** on the leaves.
4. **Gray or brown spots** on the upper leaves.
5. **Filling of the leaves** (sometimes the leaves become completely yellowed).
6. **Dark, water-soaked areas** on the stems or leaves.

These symptoms usually appear in the early stages of the disease and can progress if left untreated. Early detection and treatment are crucial to prevent further spread.


In [8]:
documents = [
    {
        "id": "tomato_early_blight",
        "crop": "Tomato",
        "disease": "Early Blight",
        "content": """
Tomato early blight is a common disease caused by the fungi
Alternaria tomatophila and Alternaria solani.

Symptoms:
Early symptoms usually appear on older leaves near the bottom of the plant.
Small dark brown spots develop on leaves. Larger spots can develop
target-like concentric rings, and the surrounding tissue may turn yellow.
Severely affected leaves can turn brown and fall from the plant.
Stem lesions may also contain concentric rings, and fruit can develop
dark lesions, often near the stem.

Disease development:
The disease can develop under moderate to warm temperatures and is
favored by wet weather, heavy dew, and high humidity.

Management:
Use disease-free seed and healthy transplants. Keep foliage as dry as
possible and prefer drip irrigation or watering at the plant base.
Improve airflow through appropriate spacing, staking, or trellising.
Remove infected leaves and plant debris, and rotate away from tomato
and related crops where appropriate.
"""
    },

    {
        "id": "tomato_late_blight",
        "crop": "Tomato",
        "disease": "Late Blight",
        "content": """
Tomato late blight is caused by Phytophthora infestans, an organism
classified as a water mold.

Symptoms:
Dark brown lesions can develop on tomato leaves and stems. Under high
humidity, white growth may appear on infected plant tissue. Tomato fruit
can develop firm, dark brown circular spots that may become soft after
secondary infections.

Disease development:
Late blight favors cool, damp conditions. Disease can spread rapidly
under favorable environmental conditions.

Management:
Scout plants regularly and remove diseased material when appropriate.
Keep foliage as dry as possible and improve air circulation. Use healthy
planting material and manage infected crop debris. Where pesticides are
used, follow the current product label and local agricultural guidance.
"""
    },

    {
        "id": "tomato_bacterial_spot",
        "crop": "Tomato",
        "disease": "Bacterial Spot",
        "content": """
Bacterial spot affects tomato and pepper plants and is caused by several
closely related Xanthomonas bacteria.

Symptoms:
Tomato leaves can develop small brown circular spots surrounded by yellow
halos. The centers of some spots may fall out and produce small holes.
The disease can also affect stems, petioles, and fruit.

Disease development:
Warm and humid conditions favor bacterial spot. The bacteria can spread
through splashing water, contaminated tools, hands, and infected plant
material.

Management:
Use healthy seed and transplants. Keep foliage dry, avoid overhead
watering when possible, improve air circulation, and sanitize tools.
Remove or bury infected crop debris after the growing season.
"""
    },

    {
        "id": "tomato_bacterial_canker",
        "crop": "Tomato",
        "disease": "Bacterial Canker",
        "content": """
Tomato bacterial canker is caused by Clavibacter michiganensis
subspecies michiganensis.

Symptoms:
Symptoms can include wilting, yellowing leaf edges, brown leaf
discoloration, stem cankers, and vascular discoloration. Fruit can
develop small raised spots sometimes called bird's-eye spots.

Disease development:
The pathogen can spread through infected seed or transplants and can
also spread through hands, equipment, and pruning activities.

Management:
Use healthy seed and transplants. Avoid working with plants when foliage
is wet. Sanitize pruning tools and equipment. Remove infected plants and
plant debris where appropriate and use crop rotation.
"""
    },

    {
        "id": "potato_early_blight",
        "crop": "Potato",
        "disease": "Early Blight",
        "content": """
Potato early blight is caused by Alternaria species including
Alternaria solani.

Symptoms:
Dark brown leaf spots can develop, often with concentric rings.
Severe infection can cause leaf death. Tubers can also be affected.

Management:
Use healthy planting material, maintain good plant vigor, improve
air circulation, avoid prolonged leaf wetness, remove infected debris,
and use appropriate crop rotation practices.
"""
    },

    {
        "id": "potato_late_blight",
        "crop": "Potato",
        "disease": "Late Blight",
        "content": """
Potato late blight is caused by Phytophthora infestans.

Symptoms:
Leaves can develop dark lesions and under humid conditions white growth
may appear. Potato tubers can develop brown, red, or purple discoloration
and may subsequently be affected by soft rot bacteria.

Disease development:
Cool and damp conditions favor disease development.

Management:
Use certified healthy potato seed, control volunteer potatoes and cull
piles, scout regularly, and remove infected material when appropriate.
Keep foliage dry where practical and follow local disease-management
guidance.
"""
    },

    {
        "id": "pepper_bacterial_spot",
        "crop": "Pepper",
        "disease": "Bacterial Spot",
        "content": """
Pepper bacterial spot is associated with Xanthomonas bacteria.

Symptoms:
Pepper leaves may develop small brown circular spots. Fruit can develop
raised corky brown spots. Symptoms are often more severe on lower leaves.

Disease development:
Warm and wet conditions favor the disease. Bacteria can spread through
splashing water, tools, hands, and infected plant material.

Management:
Use healthy seed and transplants, avoid unnecessary handling of wet
plants, keep foliage dry, improve spacing and air circulation, and
sanitize tools.
"""
    },

    {
        "id": "pepper_phytophthora_blight",
        "crop": "Pepper",
        "disease": "Phytophthora Blight",
        "content": """
Phytophthora blight can affect pepper plants and is caused by
Phytophthora capsici.

Symptoms:
Water-soaked to tan lesions can develop on leaves. Fruit can develop
soft water-soaked spots. Dark sunken lesions can occur on stems and
plants can wilt.

Disease development:
The disease can spread rapidly under favorable wet conditions.

Management:
Improve drainage and avoid conditions that keep plants excessively wet.
Use appropriate crop rotation and sanitation practices and follow local
disease-management recommendations.
"""
    },

    {
        "id": "apple_scab",
        "crop": "Apple",
        "disease": "Apple Scab",
        "content": """
Apple scab is a fungal disease that affects apple and related plants.

Symptoms:
Symptoms can include olive-green to brown lesions on leaves and fruit.
Severe infection can cause leaf loss and fruit blemishes.

Management:
Remove fallen infected leaves where practical, maintain good orchard
sanitation, improve airflow, and use disease-resistant varieties where
available. Fungicide decisions should follow local recommendations and
product labels.
"""
    },

    {
        "id": "corn_northern_leaf_blight",
        "crop": "Corn",
        "disease": "Northern Leaf Blight",
        "content": """
Northern leaf blight is a fungal disease of corn.

Symptoms:
Long, elliptical gray-green to tan lesions can develop on corn leaves.
Under favorable conditions, lesions can enlarge and cause significant
leaf area to become diseased.

Management:
Use resistant hybrids where available, rotate crops when appropriate,
manage crop residue, and follow locally recommended disease-management
practices.
"""
    }
]

print("Number of documents:", len(documents))

Number of documents: 10


Now we'll convert the documents into embeddings.

In [9]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!


For example:

"Tomato early blight causes dark leaf spots"
                      ↓
       [0.12, -0.08, 0.31, ...]

The actual vector has hundreds of dimensions.

3.4 Create document embeddings

In [10]:
document_texts = [
    f"Crop: {doc['crop']}\n"
    f"Disease: {doc['disease']}\n"
    f"{doc['content']}"
    for doc in documents
]

document_embeddings = embedding_model.encode(
    document_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embedding shape:", document_embeddings.shape)

Embedding shape: (10, 384)


because MiniLM produces 384-dimensional embeddings.

3.5 Create our FAISS vector database

In [11]:
import faiss
import numpy as np

dimension = document_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(document_embeddings.astype("float32"))

print("Documents stored in FAISS:", index.ntotal)

Documents stored in FAISS: 10


3.6 Now comes the important part: retrieval

Let's ask:

In [12]:
query = "What are the symptoms of tomato early blight?"

Convert it into an embedding:

In [13]:
query_embedding = embedding_model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
)

Search:

In [14]:
k = 3

scores, indices = index.search(
    query_embedding.astype("float32"),
    k
)

print("Retrieved documents:\n")

for rank, idx in enumerate(indices[0]):
    print(f"{rank + 1}. {documents[idx]['crop']} - {documents[idx]['disease']}")
    print(f"   Score: {scores[0][rank]:.4f}")
    print()

Retrieved documents:

1. Tomato - Early Blight
   Score: 0.7173

2. Tomato - Late Blight
   Score: 0.7053

3. Potato - Early Blight
   Score: 0.5737



We should see Tomato — Early Blight near the top.

STEP 4 — Connect retrieval to the LLM , 4.1 Create a retrieval function

In [15]:
def retrieve_documents(query, k=3):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(
        query_embedding.astype("float32"),
        k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "score": float(score),
            "document": documents[idx]
        })

    return results

Now Test it,

In [16]:
query = "What are the symptoms of tomato early blight?"

results = retrieve_documents(query, k=3)

for result in results:
    doc = result["document"]

    print(
        f"{doc['crop']} - {doc['disease']} "
        f"(score: {result['score']:.4f})"
    )

Tomato - Early Blight (score: 0.7173)
Tomato - Late Blight (score: 0.7053)
Potato - Early Blight (score: 0.5737)


4.2 Build the context (Now we take the retrieved documents and combine them into a piece of text.)

In [17]:
def build_context(results):
    context_parts = []

    for result in results:
        doc = result["document"]

        context_parts.append(
            f"Crop: {doc['crop']}\n"
            f"Disease: {doc['disease']}\n"
            f"{doc['content']}"
        )

    return "\n\n---\n\n".join(context_parts)

Test it,You should see the information from the three retrieved documents.

In [18]:
context = build_context(results)

print(context)

Crop: Tomato
Disease: Early Blight

Tomato early blight is a common disease caused by the fungi
Alternaria tomatophila and Alternaria solani.

Symptoms:
Early symptoms usually appear on older leaves near the bottom of the plant.
Small dark brown spots develop on leaves. Larger spots can develop
target-like concentric rings, and the surrounding tissue may turn yellow.
Severely affected leaves can turn brown and fall from the plant.
Stem lesions may also contain concentric rings, and fruit can develop
dark lesions, often near the stem.

Disease development:
The disease can develop under moderate to warm temperatures and is
favored by wet weather, heavy dew, and high humidity.

Management:
Use disease-free seed and healthy transplants. Keep foliage as dry as
possible and prefer drip irrigation or watering at the plant base.
Improve airflow through appropriate spacing, staking, or trellising.
Remove infected leaves and plant debris, and rotate away from tomato
and related crops where appro

4.3 Now create the RAG prompt

This is one of the most important pieces of the entire project.

In [19]:
prompt = f"""
You are an agricultural disease assistant.

Answer the user's question using only the provided context.

If the answer is not available in the context, say:
"I don't have enough information in my knowledge base to answer that."

Do not invent facts.

Context:
{context}

User question:
{query}

Answer:
"""

print(prompt)


You are an agricultural disease assistant.

Answer the user's question using only the provided context.

If the answer is not available in the context, say:
"I don't have enough information in my knowledge base to answer that."

Do not invent facts.

Context:
Crop: Tomato
Disease: Early Blight

Tomato early blight is a common disease caused by the fungi
Alternaria tomatophila and Alternaria solani.

Symptoms:
Early symptoms usually appear on older leaves near the bottom of the plant.
Small dark brown spots develop on leaves. Larger spots can develop
target-like concentric rings, and the surrounding tissue may turn yellow.
Severely affected leaves can turn brown and fall from the plant.
Stem lesions may also contain concentric rings, and fruit can develop
dark lesions, often near the stem.

Disease development:
The disease can develop under moderate to warm temperatures and is
favored by wet weather, heavy dew, and high humidity.

Management:
Use disease-free seed and healthy transplan

4.4 Send the RAG prompt to Qwen

Now reuse our model.

In [20]:
messages = [
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=250,
    temperature=0.3,
    top_p=0.8,
    do_sample=True
)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(response)

Early symptoms of tomato early blight include small dark brown spots on leaves, larger target-like concentric rings, yellowing of surrounding tissue, and brown and falling leaves.


Why did I lower temperature from 0.7 to 0.3?

For our domain-specific factual assistant, we don't need highly creative answers.

Lower temperature generally makes generation more deterministic.

So:

0.7 → more variation
0.3 → more conservative/deterministic

Don't describe temperature as a strict "accuracy control" in the interview. It's a generation-sampling parameter.

**STEP 4.5 — Turn it into one function**

Once the above works, we're going to make our actual RAG pipeline:

In [21]:
def ask_agri_llm(query, k=3):
    # 1. Retrieve
    results = retrieve_documents(query, k)

    # 2. Build context
    context = build_context(results)

    # 3. Create prompt
    prompt = f"""
You are an agricultural disease assistant.

Answer the user's question using only the provided context.

If the answer is not available in the context, say:
"I don't have enough information in my knowledge base to answer that."

Do not invent facts.

Context:
{context}

User question:
{query}

Answer:
"""

    # 4. Generate
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        temperature=0.3,
        top_p=0.8,
        do_sample=True
    )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return answer, results

In [22]:
answer, retrieved = ask_agri_llm(
    "What are the symptoms of tomato early blight?"
)

print(answer)

Early symptoms of tomato early blight are small dark brown spots on leaves, larger target-like concentric rings, yellowing of surrounding tissue, and brown and falling leaves.


**Now test something important**
Don't only test the question we designed the knowledge base around.

In [23]:
questions = [
    "What are the symptoms of tomato early blight?",
    "How can tomato early blight be managed?",
    "What causes potato late blight?",
    "What are the symptoms of apple scab?",
    "What is the best treatment for rice blast?"
]

for question in questions:
    print("=" * 80)
    print("QUESTION:", question)

    answer, retrieved = ask_agri_llm(question)

    print("\nANSWER:")
    print(answer)

    print("\nRETRIEVED:")
    for item in retrieved:
        doc = item["document"]
        print(
            f"- {doc['crop']} / {doc['disease']} "
            f"({item['score']:.3f})"
        )

    print()

QUESTION: What are the symptoms of tomato early blight?

ANSWER:
The symptoms of tomato early blight are small dark brown spots on leaves, larger target-like concentric rings, yellowing of surrounding tissue, and brown and falling leaves.

RETRIEVED:
- Tomato / Early Blight (0.717)
- Tomato / Late Blight (0.705)
- Potato / Early Blight (0.574)

QUESTION: How can tomato early blight be managed?

ANSWER:
Use disease-free seed and healthy transplants. Keep foliage as dry as possible and prefer drip irrigation or watering at the plant base. Improve airflow through appropriate spacing, staking, or trellising. Remove infected leaves and plant debris, and rotate away from tomato and related crops where appropriate.

RETRIEVED:
- Tomato / Late Blight (0.689)
- Tomato / Early Blight (0.675)
- Potato / Late Blight (0.509)

QUESTION: What causes potato late blight?

ANSWER:
Potato late blight is caused by Phytophthora infestans, an organism classified as a water mold.

RETRIEVED:
- Potato / Early

The rice blast question is intentional.

We don't have rice blast in our knowledge base.

We want to see whether our model follows:

"I don't have enough information in my knowledge base to answer that."

This gives us our first test of hallucination control.

**STEP 5.1 — Create our fine-tuning dataset**
We're going to create instruction/response examples.

In [24]:
from datasets import Dataset

training_data = [
    {
        "instruction": "What are the symptoms of tomato early blight?",
        "response": "Tomato early blight commonly causes small dark brown spots on older leaves. Larger lesions may develop concentric rings, and the surrounding leaf tissue can turn yellow. Severely affected leaves may turn brown and fall."
    },
    {
        "instruction": "What causes tomato early blight?",
        "response": "Tomato early blight is caused by fungi including Alternaria tomatophila and Alternaria solani."
    },
    {
        "instruction": "How can tomato early blight be managed?",
        "response": "Management includes using healthy planting material, keeping foliage as dry as possible, improving air circulation, removing infected leaves and plant debris, and using appropriate crop rotation practices."
    },
    {
        "instruction": "What are the symptoms of tomato late blight?",
        "response": "Tomato late blight can cause dark brown lesions on leaves and stems. Under humid conditions, white growth may appear on infected tissue. Fruit can develop firm dark brown lesions."
    },
    {
        "instruction": "What causes tomato late blight?",
        "response": "Tomato late blight is caused by Phytophthora infestans."
    },
    {
        "instruction": "How can tomato late blight be managed?",
        "response": "Management includes using healthy planting material, monitoring plants regularly, keeping foliage dry where practical, improving air circulation, and following local disease-management recommendations."
    },
    {
        "instruction": "What are the symptoms of potato early blight?",
        "response": "Potato early blight can cause dark brown leaf spots, often with concentric rings. Severe infection can cause leaf death, and tubers can also be affected."
    },
    {
        "instruction": "What causes potato late blight?",
        "response": "Potato late blight is caused by Phytophthora infestans."
    },
    {
        "instruction": "What are the symptoms of pepper bacterial spot?",
        "response": "Pepper bacterial spot can cause small brown circular spots on leaves and raised corky brown spots on fruit. Symptoms can be more severe on lower leaves."
    },
    {
        "instruction": "What is apple scab?",
        "response": "Apple scab is a fungal disease that affects apple and related plants. It can produce olive-green to brown lesions on leaves and fruit and may cause fruit blemishes and leaf loss."
    },
    {
        "instruction": "What are the symptoms of corn northern leaf blight?",
        "response": "Northern leaf blight can produce long elliptical gray-green to tan lesions on corn leaves. Under favorable conditions, lesions can enlarge and affect significant leaf area."
    },
    {
        "instruction": "Why is crop sanitation important for disease management?",
        "response": "Crop sanitation can reduce sources of infection by removing infected plant material and crop debris. It should be combined with other appropriate disease-management practices."
    }
]

dataset = Dataset.from_list(training_data)

print(dataset)

Dataset({
    features: ['instruction', 'response'],
    num_rows: 12
})


**STEP 5.2 — Format the dataset**

In [25]:
def format_example(example):
    return {
        "messages": [
            {
                "role": "user",
                "content": example["instruction"]
            },
            {
                "role": "assistant",
                "content": example["response"]
            }
        ]
    }

formatted_dataset = dataset.map(format_example)

print(formatted_dataset[0])

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

{'instruction': 'What are the symptoms of tomato early blight?', 'response': 'Tomato early blight commonly causes small dark brown spots on older leaves. Larger lesions may develop concentric rings, and the surrounding leaf tissue can turn yellow. Severely affected leaves may turn brown and fall.', 'messages': [{'content': 'What are the symptoms of tomato early blight?', 'role': 'user'}, {'content': 'Tomato early blight commonly causes small dark brown spots on older leaves. Larger lesions may develop concentric rings, and the surrounding leaf tissue can turn yellow. Severely affected leaves may turn brown and fall.', 'role': 'assistant'}]}


**STEP 5.3 — Load LoRA

Now we're going to use PEFT.**

In [26]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    task_type="CAUSAL_LM"
)

print(lora_config)

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.21.1', base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules={'v_proj', 'o_proj', 'q_proj', 'k_proj'}, exclude_modules=None, lora_alpha=16, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, kasa_config=None, ensure_weight_tying=False)


STEP 5.4 — Before training, let's inspect the model

In [28]:
!pip install -q -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 51.6 MB/s eta 0:00:00


In [29]:
from peft import get_peft_model

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

trainable params: 2,293,760 || all params: 598,343,680 || trainable%: 0.3834


In [30]:
import torch
import peft
import torchao

print("PyTorch:", torch.__version__)
print("PEFT:", peft.__version__)
print("TorchAO:", torchao.__version__)

PyTorch: 2.11.0+cu128
PEFT: 0.21.1
TorchAO: 0.18.0


Step 6 — Fine-tune with SFT

In [31]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./agri_llm_lora",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    fp16=True,
    report_to="none",
    max_length=512
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted_dataset,
    processing_class=tokenizer
)

trainer.train()

Tokenizing train dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
1,4.221134
2,3.998910
3,3.451837
4,2.938774
5,3.136807
6,3.110304
7,2.655864
8,2.336997
9,2.753506
10,2.383661


TrainOutput(global_step=15, training_loss=2.8122377395629883, metrics={'train_runtime': 18.2437, 'train_samples_per_second': 3.289, 'train_steps_per_second': 0.822, 'total_flos': 9096086421504.0, 'train_loss': 2.8122377395629883, 'epoch': 5.0})

One important point

Our dataset is intentionally small for today's interview project. Do not describe this as a production-grade fine-tuning dataset.

In the interview, say:

"I created a small supervised dataset to demonstrate domain adaptation using LoRA. The goal was to understand and implement the complete fine-tuning pipeline rather than claim production-level model performance."

**Step 7 — Save the fine-tuned LoRA adapter**

In [32]:
save_path = "./agri_llm_lora/final_adapter"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("LoRA adapter saved to:", save_path)

LoRA adapter saved to: ./agri_llm_lora/final_adapter


Then verify,

In [33]:
import os

print(os.listdir(save_path))

['adapter_config.json', 'README.md', 'adapter_model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'chat_template.jinja']


**Step 8 — Test the LoRA fine-tuned model**

In [34]:
def ask_finetuned_model(query):
    messages = [
        {
            "role": "user",
            "content": query
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.3,
        top_p=0.8,
        do_sample=True
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response

Then test these

In [35]:
print("Q1:")
print(ask_finetuned_model(
    "What are the symptoms of tomato early blight?"
))

print("\nQ2:")
print(ask_finetuned_model(
    "What causes tomato late blight?"
))

print("\nQ3:")
print(ask_finetuned_model(
    "How can tomato early blight be managed?"
))

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in Qwen3DecoderLayer. Setting `past_key_values=None`.


Q1:
Tom Instructions InstructionsQuestion Instructions Instructions Instructions Instructions Instructionslicants Instructions Instructions Instructions Instructions Instructions Instructions Instructionslicants Question Instructions Instructions InstructionslicantslicantslicantslicantsClassQuestion AnswerQuestionQuestion QuestionQuestion Instructions InstructionsQuestion Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructions Instructionslicantslicants Instructions Instructions Instructions Instructions InstructionslicantsClassQuestion Instructionslicantslicants Answer Answer Answer Instructions Instructions Instructions Instructions Instructions Instructions Instructionslicantslicants Instructions Instructions Instructions InstructionsQuestion QuestionQuestion QuestionQuestionQuestion Instructions Instructionslicants Instructionslicants Instructionslicantsropriate Question Instructions Instructions Instructions

1. Reload the original Qwen model

In [36]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Fresh base model loaded.")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Fresh base model loaded.


In [37]:
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

2.Recreate Lora

In [38]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 2,293,760 || all params: 598,343,680 || trainable%: 0.3834


3. Train much more gently

In [39]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./agri_llm_lora_v2",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    logging_steps=1,
    save_strategy="epoch",
    fp16=True,
    report_to="none",
    max_length=512
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted_dataset,
    processing_class=tokenizer
)

trainer.train()

Tokenizing train dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
1,4.221134
2,4.239590
3,3.926778
4,3.691504
5,3.987108
6,3.957819


TrainOutput(global_step=6, training_loss=4.003988901774089, metrics={'train_runtime': 12.196, 'train_samples_per_second': 1.968, 'train_steps_per_second': 0.492, 'total_flos': 3639497195520.0, 'train_loss': 4.003988901774089, 'epoch': 2.0})

**1. Make sure inference mode is clean**

In [40]:
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 1024)
        (layers): ModuleList(
          (0-27): 28 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1024, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(


2. Test the fine-tuned model

In [41]:
def ask_finetuned_model(query):
    messages = [
        {
            "role": "user",
            "content": query
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.2,
            top_p=0.8,
            do_sample=True
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response

Now Test,

In [42]:
print("Q1:", ask_finetuned_model(
    "What causes tomato late blight?"
))

print("\nQ2:", ask_finetuned_model(
    "What are the symptoms of tomato early blight?"
))

print("\nQ3:", ask_finetuned_model(
    "How can tomato early blight be managed?"
))

Q1: Tomato late blight is a fungal disease caused by the *Phytophthora infestans* pathogen. It primarily affects the late blight stage of the tomato plant, which occurs when the fruit is still in the early stages of ripening. The disease spreads through water, especially from infected leaves, and can cause the fruit to develop spots, blemishes, or blackened areas. It is a significant threat to tomato crops, particularly in regions with poor soil quality or high humidity.

Q2: Tomato early blight, also known as **tomato leaf spot disease**, is a fungal infection that affects the leaves of tomato plants. The symptoms typically include:

1. **Leaf spots**: Small, circular or irregularly shaped spots on the leaves, often with a yellowish or brownish color.
2. **Mosaic-like patterns**: In some cases, the spots may form a mosaic pattern across the leaves.
3. **Mild to moderate leaf damage**: The affected leaves may show slight discoloration or curling.
4. **Leaf curling**: In severe cases, t

**Step 9 — Connect your existing RAG to the LoRA model** You already have retrieve_documents() and build_context(). We'll reuse them.

In [43]:
def ask_rag_lora(query, k=3):
    # 1. Retrieve relevant documents
    results = retrieve_documents(query, k=k)

    # 2. Build context
    context = build_context(results)

    # 3. Create grounded prompt
    prompt = f"""
You are an agricultural disease assistant.

Answer the user's question using only the provided context.

Rules:
- Use only information supported by the context.
- Do not invent facts.
- If the context does not contain enough information, say:
"I don't have enough information in my knowledge base to answer that."
- Keep the answer concise and clear.

Context:
{context}

User question:
{query}

Answer:
"""

    # 4. Apply Qwen chat template
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    # 5. Tokenize
    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    # 6. Generate using LoRA-adapted model
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.2,
            top_p=0.8,
            do_sample=True
        )

    # 7. Decode only newly generated tokens
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response, results

Test it,

In [44]:
answer, results = ask_rag_lora(
    "What are the symptoms of tomato early blight?"
)

print("ANSWER:")
print(answer)

print("\nRETRIEVED DOCUMENTS:")
for r in results:
    doc = r["document"]
    print(
        f"- {doc['crop']} / {doc['disease']} "
        f"(score: {r['score']:.4f})"
    )

ANSWER:
Small dark brown spots develop on leaves. Larger spots can develop target-like concentric rings, and the surrounding tissue may turn yellow. Severely affected leaves can turn brown and fall from the plant.

RETRIEVED DOCUMENTS:
- Tomato / Early Blight (score: 0.7173)
- Tomato / Late Blight (score: 0.7053)
- Potato / Early Blight (score: 0.5737)


**Step 10 — Create an evaluation set**

In [45]:
evaluation_data = [
    {
        "question": "What are the symptoms of tomato early blight?",
        "expected_disease": "Early Blight",
        "keywords": ["dark brown spots", "concentric rings", "yellow", "leaves"]
    },
    {
        "question": "What causes tomato late blight?",
        "expected_disease": "Late Blight",
        "keywords": ["Phytophthora infestans"]
    },
    {
        "question": "What are the symptoms of potato early blight?",
        "expected_disease": "Early Blight",
        "keywords": ["dark brown", "concentric rings", "leaf"]
    },
    {
        "question": "What is apple scab?",
        "expected_disease": "Apple Scab",
        "keywords": ["fungal", "olive-green", "brown", "lesions"]
    },
    {
        "question": "What are the symptoms of corn northern leaf blight?",
        "expected_disease": "Northern Leaf Blight",
        "keywords": ["elliptical", "gray-green", "lesions", "leaves"]
    },
    {
        "question": "What are the symptoms of pepper bacterial spot?",
        "expected_disease": "Bacterial Spot",
        "keywords": ["brown", "spots", "fruit", "leaves"]
    }
]

print("Evaluation examples:", len(evaluation_data))

Evaluation examples: 6


**Step 11 — Measure retrieval Recall@3**  This checks whether the correct disease document appears somewhere in the top 3 retrieved documents.

In [46]:
def evaluate_retrieval(eval_data, k=3):
    hits = 0

    for item in eval_data:
        results = retrieve_documents(item["question"], k=k)

        retrieved_diseases = [
            r["document"]["disease"]
            for r in results
        ]

        hit = item["expected_disease"] in retrieved_diseases

        if hit:
            hits += 1

        print("\nQuestion:", item["question"])
        print("Expected:", item["expected_disease"])
        print("Retrieved:", retrieved_diseases)
        print("Hit:", hit)

    recall = hits / len(eval_data)

    print("\n==============================")
    print(f"Recall@{k}: {recall:.2%}")
    print("==============================")

    return recall

In [47]:
retrieval_recall = evaluate_retrieval(evaluation_data, k=3)


Question: What are the symptoms of tomato early blight?
Expected: Early Blight
Retrieved: ['Early Blight', 'Late Blight', 'Early Blight']
Hit: True

Question: What causes tomato late blight?
Expected: Late Blight
Retrieved: ['Late Blight', 'Early Blight', 'Early Blight']
Hit: True

Question: What are the symptoms of potato early blight?
Expected: Early Blight
Retrieved: ['Early Blight', 'Late Blight', 'Early Blight']
Hit: True

Question: What is apple scab?
Expected: Apple Scab
Retrieved: ['Apple Scab', 'Bacterial Canker', 'Late Blight']
Hit: True

Question: What are the symptoms of corn northern leaf blight?
Expected: Northern Leaf Blight
Retrieved: ['Northern Leaf Blight', 'Early Blight', 'Early Blight']
Hit: True

Question: What are the symptoms of pepper bacterial spot?
Expected: Bacterial Spot
Retrieved: ['Bacterial Spot', 'Bacterial Spot', 'Phytophthora Blight']
Hit: True

Recall@3: 100.00%


**Step 12 — Test an out-of-knowledge question**

Remember earlier we saw that the retriever still returns nearest documents even for an unrelated question. That's normal for basic top-K vector search.

In [48]:
answer, results = ask_rag_lora(
    "What are the symptoms of rice blast?"
)

print("ANSWER:")
print(answer)

print("\nTOP RETRIEVAL SCORES:")
for r in results:
    print(
        f"{r['document']['crop']} / "
        f"{r['document']['disease']} "
        f"-> {r['score']:.4f}"
    )

ANSWER:
I don't have enough information in my knowledge base to answer that.

TOP RETRIEVAL SCORES:
Tomato / Bacterial Canker -> 0.3771
Pepper / Phytophthora Blight -> 0.3627
Tomato / Late Blight -> 0.3421


**Step 13 — Add retrieval confidence threshold**

In [49]:
def retrieve_documents_with_threshold(query, k=3, threshold=0.5):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(
        query_embedding.astype("float32"),
        k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if score >= threshold:
            results.append({
                "score": float(score),
                "document": documents[idx]
            })

    return results

Now test both a known and unknown question:

In [50]:
known_results = retrieve_documents_with_threshold(
    "What are the symptoms of tomato early blight?"
)

unknown_results = retrieve_documents_with_threshold(
    "What are the symptoms of rice blast?"
)

print("KNOWN QUESTION:")
for r in known_results:
    print(
        r["document"]["disease"],
        "->",
        round(r["score"], 4)
    )

print("\nUNKNOWN QUESTION:")
for r in unknown_results:
    print(
        r["document"]["disease"],
        "->",
        round(r["score"], 4)
    )

KNOWN QUESTION:
Early Blight -> 0.7173
Late Blight -> 0.7053
Early Blight -> 0.5737

UNKNOWN QUESTION:


**Step 14 — Update the final RAG + LoRA function**

In [51]:
def ask_rag_lora(query, k=3, threshold=0.5):

    # Retrieve relevant documents
    results = retrieve_documents_with_threshold(
        query,
        k=k,
        threshold=threshold
    )

    # No sufficiently relevant information
    if not results:
        return (
            "I don't have enough information in my knowledge base "
            "to answer that.",
            []
        )

    # Build context
    context = build_context(results)

    prompt = f"""
You are an agricultural disease assistant.

Answer the user's question using only the provided context.

Rules:
- Use only information supported by the context.
- Do not invent facts.
- Keep the answer concise and clear.

Context:
{context}

User question:
{query}

Answer:
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.2,
            top_p=0.8,
            do_sample=True
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response, results

Test the final behavior

In [52]:
answer, results = ask_rag_lora(
    "What are the symptoms of rice blast?"
)

print(answer)
print("\nRetrieved documents:", len(results))

I don't have enough information in my knowledge base to answer that.

Retrieved documents: 0


**Step 15 — Base Qwen function**

In [53]:
def ask_base_model(query):
    messages = [
        {
            "role": "user",
            "content": query
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    # Temporarily disable the LoRA adapter
    model.disable_adapter_layers()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.2,
            top_p=0.8,
            do_sample=True
        )

    model.enable_adapter_layers()

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response

Test

In [54]:
print(ask_base_model(
    "What are the symptoms of tomato early blight?"
))

Tomato early blight, also known as **tomato bacterial wilt**, is a disease caused by the bacterium *Corynebacterium aphthaeum*. It primarily affects the **lower parts of the tomato plant**, particularly the **stem, leaves, and fruit**. Here are the common symptoms:

### 1. **Leaf and Stem Issues**
- **Yellowing or discoloration** of the leaves, especially in the lower leaves.
- **Wilting** of the lower parts of the plant.
- **Browning** or **blackening** of the stem or leaves.
- **Fruit rot** or **deceleration** of the fruit.

### 2. **Fruit Symptoms**
- **Yellow


**Step 16 — RAG + Base Qwen** We already have our RAG function, but it currently uses the LoRA adapter.

Create a base-model version:

In [55]:
def ask_rag_base(query, k=3, threshold=0.5):

    results = retrieve_documents_with_threshold(
        query,
        k=k,
        threshold=threshold
    )

    if not results:
        return (
            "I don't have enough information in my knowledge base to answer that.",
            []
        )

    context = build_context(results)

    prompt = f"""
You are an agricultural disease assistant.

Answer the user's question using only the provided context.

Rules:
- Use only information supported by the context.
- Do not invent facts.
- Keep the answer concise and clear.

Context:
{context}

User question:
{query}

Answer:
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    model.disable_adapter_layers()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.2,
            top_p=0.8,
            do_sample=True
        )

    model.enable_adapter_layers()

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response, results

**Step 17 — Compare all three**  Let's use one question first:

In [56]:
test_question = "What are the symptoms of tomato early blight?"

print("=" * 70)
print("BASE QWEN")
print("=" * 70)
print(ask_base_model(test_question))

print("\n" + "=" * 70)
print("RAG + BASE QWEN")
print("=" * 70)

rag_base_answer, _ = ask_rag_base(test_question)
print(rag_base_answer)

print("\n" + "=" * 70)
print("RAG + LoRA")
print("=" * 70)

rag_lora_answer, _ = ask_rag_lora(test_question)
print(rag_lora_answer)

BASE QWEN
Tomato early blight, also known as **tomato bacterial wilt**, is a fungal disease that affects tomato plants. It is caused by the bacterium *Corynebacterium tomato* and spreads through water and air. The symptoms of early blight include:

1. **Yellowing and wilting of leaves** — especially in the early stages.
2. **Blossom points** — small, yellow spots on the leaves.
3. **Blossom streaks** — dark, streaked areas on the leaves.
4. **Mosaic-like yellowing** — a widespread yellowing of the leaves.
5. **Wilting and curling** — the leaves may curl or turn yellow.
6. **Bloss

RAG + BASE QWEN
Small dark brown spots develop on leaves, larger spots can develop target-like concentric rings, and the surrounding tissue may turn yellow. Severely affected leaves can turn brown and fall from the plant. Stem lesions may also contain concentric rings, and fruit can develop dark lesions, often near the stem.

RAG + LoRA
Small dark brown spots develop on leaves, larger spots can develop target

**Step 18 — Save the final adapter**

In [57]:
final_adapter_path = "./agri_llm_lora_final"

model.save_pretrained(final_adapter_path)
tokenizer.save_pretrained(final_adapter_path)

print("Final adapter saved successfully.")

Final adapter saved successfully.


In [58]:
import os

print(os.listdir(final_adapter_path))

['adapter_config.json', 'README.md', 'adapter_model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'chat_template.jinja']


The important files are:

adapter_config.json |
adapter_model.safetensors

adapter_model.safetensors contains your trained LoRA weights, while adapter_config.json tells PEFT how those weights should be attached to Qwen.

So your fine-tuning portion is complete.

**Step 19 — Save your knowledge base**

We should now save the 10 agricultural documents so your RAG system isn't dependent on notebook variables.

In [59]:
import json
import os

os.makedirs("./data", exist_ok=True)

with open("./data/knowledge_base.json", "w", encoding="utf-8") as f:
    json.dump(documents, f, indent=2, ensure_ascii=False)

print("Knowledge base saved.")
print("Documents:", len(documents))

Knowledge base saved.
Documents: 10


In [60]:
import os

print(os.path.getsize("./data/knowledge_base.json"), "bytes")

7348 bytes


**Step 20 — Save your evaluation results**

In [61]:
evaluation_results = {
    "model": "Qwen/Qwen3-0.6B",
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "vector_database": "FAISS",
    "fine_tuning": "LoRA",
    "knowledge_base_documents": len(documents),
    "evaluation_questions": len(evaluation_data),
    "retrieval_metric": "Recall@3",
    "retrieval_recall": retrieval_recall,
    "similarity_threshold": 0.5
}

with open("./evaluation_results.json", "w", encoding="utf-8") as f:
    json.dump(evaluation_results, f, indent=2)

print(json.dumps(evaluation_results, indent=2))

{
  "model": "Qwen/Qwen3-0.6B",
  "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
  "vector_database": "FAISS",
  "fine_tuning": "LoRA",
  "knowledge_base_documents": 10,
  "evaluation_questions": 6,
  "retrieval_metric": "Recall@3",
  "retrieval_recall": 1.0,
  "similarity_threshold": 0.5
}


**Step 21 — Build the Streamlit app**
Now let's turn the notebook work into something you can actually demonstrate.

For today, we'll make a single-file Streamlit app that contains:

Your 10-document knowledge base,
Sentence Transformer embeddings,
FAISS retrieval,
Qwen3-0.6B,
LoRA adapter,
Similarity threshold,
Chat interface,

**First, install Streamlit**

In [62]:
!pip install -q streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 53.5 MB/s eta 0:00:00


In [67]:
%%writefile app.py

import streamlit as st
import json
import numpy as np
import faiss
import torch

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel


# =========================
# Configuration
# =========================

BASE_MODEL = "Qwen/Qwen3-0.6B"
ADAPTER_PATH = "./agri_llm_lora_final"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

THRESHOLD = 0.5


# =========================
# Page Configuration
# =========================

st.set_page_config(
    page_title="AgriLLM",
    page_icon="🌱",
    layout="wide"
)

st.title("🌱 AgriLLM")
st.subheader("Crop Disease Knowledge Assistant")

st.write(
    "Ask questions about crop diseases using a "
    "RAG + LoRA powered language model."
)


# =========================
# Load Knowledge Base
# =========================

@st.cache_data
def load_documents():

    with open(
        "./data/knowledge_base.json",
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(f)


documents = load_documents()


# =========================
# Load Embedding Model
# =========================

@st.cache_resource
def load_embedding_model():

    return SentenceTransformer(
        EMBEDDING_MODEL
    )


embedding_model = load_embedding_model()


# =========================
# Create FAISS Index
# =========================

@st.cache_resource
def create_faiss_index(documents):

    texts = [
        f"{doc['crop']} {doc['disease']} {doc['content']}"
        for doc in documents
    ]

    embeddings = embedding_model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatIP(dimension)

    index.add(
        embeddings.astype("float32")
    )

    return index


index = create_faiss_index(documents)


# =========================
# Load Qwen + LoRA
# =========================

@st.cache_resource
def load_llm():

    tokenizer = AutoTokenizer.from_pretrained(
        BASE_MODEL
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    model = PeftModel.from_pretrained(
        base_model,
        ADAPTER_PATH
    )

    model.eval()

    return tokenizer, model


tokenizer, model = load_llm()


# =========================
# Retrieval
# =========================

def retrieve_documents(
    query,
    k=3,
    threshold=THRESHOLD
):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(
        query_embedding.astype("float32"),
        k
    )

    results = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        if score >= threshold:

            results.append({
                "score": float(score),
                "document": documents[idx]
            })

    return results


# =========================
# Build Context
# =========================

def build_context(results):

    context_parts = []

    for result in results:

        doc = result["document"]

        context_parts.append(
            f"Crop: {doc['crop']}\n"
            f"Disease: {doc['disease']}\n"
            f"{doc['content']}"
        )

    return "\n\n---\n\n".join(
        context_parts
    )


# =========================
# Generate Answer
# =========================

def generate_answer(query, context):

    prompt = f"""
You are AgriLLM, a crop disease knowledge assistant.

Answer the user's question using ONLY the
provided knowledge base context.

If the answer cannot be supported by the context,
say:

"I don't have enough information in my knowledge base to answer that."

Do not invent facts.

Knowledge Base Context:
{context}

User Question:
{query}
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.2,
            top_p=0.8,
            do_sample=True
        )

    generated_tokens = outputs[
        0
    ][inputs.input_ids.shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()


# =========================
# User Interface
# =========================

question = st.text_input(
    "Ask a crop disease question:",
    placeholder="Example: What are the symptoms of tomato early blight?"
)


if st.button("🔍 Ask"):

    if not question.strip():

        st.warning(
            "Please enter a question."
        )

    else:

        with st.spinner(
            "Searching knowledge base and generating answer..."
        ):

            results = retrieve_documents(
                question
            )

            if not results:

                answer = (
                    "I don't have enough information "
                    "in my knowledge base to answer that."
                )

            else:

                context = build_context(
                    results
                )

                answer = generate_answer(
                    question,
                    context
                )

        st.markdown("### 🤖 Answer")

        st.write(answer)

        if results:

            st.markdown(
                "### 📚 Retrieved Knowledge"
            )

            for result in results:

                doc = result["document"]

                with st.expander(
                    f"{doc['crop']} — "
                    f"{doc['disease']} "
                    f"(similarity: {result['score']:.3f})"
                ):

                    st.write(
                        doc["content"]
                    )

Overwriting app.py


In [68]:
import os

print("app.py exists:", os.path.exists("./app.py"))
print("knowledge base exists:", os.path.exists("./data/knowledge_base.json"))
print("LoRA adapter exists:", os.path.exists("./agri_llm_lora_final/adapter_model.safetensors"))

app.py exists: True
knowledge base exists: True
LoRA adapter exists: True


In [70]:
!streamlit run app.py &>/content/streamlit.log &

In [71]:
!cat /content/streamlit.log



2026-09-29 11:15:58.374 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.109.100.110:8501



In [69]:
!pkill -f streamlit

In [72]:
import torch
import transformers
import peft
import trl
import datasets
import sentence_transformers
import faiss
import accelerate

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("datasets:", datasets.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("faiss:", faiss.__version__)
print("accelerate:", accelerate.__version__)

torch: 2.11.0+cu128
transformers: 5.17.0
peft: 0.21.1
trl: 1.14.0
datasets: 5.0.1
sentence-transformers: 6.1.0
faiss: 1.15.1
accelerate: 1.15.0


**Step 1 — Create requirements.txt**

In [73]:
%%writefile requirements.txt

torch==2.11.0
transformers==5.17.0
peft==0.21.1
trl==1.14.0
datasets==5.0.1
sentence-transformers==6.1.0
faiss-cpu==1.15.1
accelerate==1.15.0
torchao==0.18.0
streamlit

Writing requirements.txt


**Step 2 — Verify the important project files**

In [75]:
import os

files_to_check = [
    "app.py",
    "requirements.txt",
    "data/knowledge_base.json",
    "evaluation_results.json",
    "agri_llm_lora_final/adapter_config.json",
    "agri_llm_lora_final/adapter_model.safetensors"
]

for file in files_to_check:
    print(f"{'✅' if os.path.exists(file) else '❌'} {file}")

✅ app.py
✅ requirements.txt
✅ data/knowledge_base.json
✅ evaluation_results.json
✅ agri_llm_lora_final/adapter_config.json
✅ agri_llm_lora_final/adapter_model.safetensors


In [76]:
%%writefile README.md

# 🌱 AgriLLM — Crop Disease Knowledge Assistant

A domain-specific AI assistant for answering crop disease questions using a **Retrieval-Augmented Generation (RAG)** pipeline combined with **LoRA fine-tuning**.

The project uses a small open-source language model, a vector search system, and a curated agricultural knowledge base to reduce hallucinations and provide grounded answers.

---

## 🚀 Project Overview

The goal of this project was to explore how Large Language Models can be adapted for a specific domain.

The project demonstrates:

- Large Language Model inference
- Retrieval-Augmented Generation (RAG)
- Semantic embeddings
- FAISS vector search
- LoRA parameter-efficient fine-tuning
- Retrieval evaluation using Recall@3
- Streamlit application development

---

## 🏗️ Architecture

```text
                    User Question
                         │
                         ▼
              Sentence Transformer
                    Embeddings
                         │
                         ▼
                  FAISS Vector DB
                         │
                    Top-K Retrieval
                         │
                         ▼
                Relevant Documents
                         │
                         ▼
                Context Construction
                         │
                         ▼
                 Qwen3-0.6B + LoRA
                         │
                         ▼
                       Answer

Writing README.md
